# EDA — World Happiness 2015-2019
**Workshop 3 · Parte A · Paso 1.** Objetivo: perfilar los 5 CSV (faltantes, duplicados, nombres, tipos, diferencias de esquema, outliers) y proponer un **esquema unificado**.

> Ejecutar desde la carpeta `notebooks/` (las rutas son relativas: `../data/raw`).

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
RAW = "../data/raw"
YEARS = [2015, 2016, 2017, 2018, 2019]
raw = {y: pd.read_csv(f"{RAW}/{y}.csv") for y in YEARS}
{y: df.shape for y, df in raw.items()}

## 1. Esquema de cada año (nombres y tipos)

In [ ]:
for y, df in raw.items():
    print(f"--- {y}: {df.shape[0]} filas x {df.shape[1]} columnas")
    print(df.dtypes.to_string(), "\n")

## 2. Diferencias de esquema entre años

In [ ]:
all_cols = sorted(set().union(*[set(df.columns) for df in raw.values()]))
schema = pd.DataFrame({y: [c in raw[y].columns for c in all_cols] for y in YEARS}, index=all_cols)
schema.replace({True: "X", False: ""})

**Observaciones**
- El mismo concepto tiene **nombres distintos**: `Country` / `Country or region`; `Happiness Score` / `Happiness.Score` / `Score`; `Economy (GDP per Capita)` / `Economy..GDP.per.Capita.` / `GDP per capita`; `Family` / `Social support`; `Health (Life Expectancy)` / `Healthy life expectancy`; `Freedom` / `Freedom to make life choices`; `Trust (Government Corruption)` / `Perceptions of corruption`.
- 2015-2017 traen columnas que 2018-2019 no tienen (`Region`, `Standard Error`, intervalos/whiskers, `Dystopia Residual`).
- 2017 usa puntos en vez de espacios y paréntesis, y cambia el **orden** de `Generosity` y `Trust`.
- Las columnas comunes a los 5 años son: país, rank, score, GDP, family/social support, health, freedom, corruption, generosity.

## 3. Valores faltantes

In [ ]:
for y, df in raw.items():
    miss = df.isna().sum(); miss = miss[miss > 0]
    print(y, "->", miss.to_dict() if len(miss) else "sin faltantes")
print()
print(raw[2018][raw[2018].isna().any(axis=1)].to_string())

**Observación:** solo 2018 tiene un faltante (`Perceptions of corruption` de *United Arab Emirates*). Es 1 fila de ~780: se **elimina** esa fila (ver decisiones en el notebook de entrenamiento); imputar un valor inventaría un dato de corrupción para un país sin información, y el costo de perderla es mínimo.

## 4. Duplicados (filas completas y país repetido dentro de un mismo año)

In [ ]:
def country_col(df): return "Country" if "Country" in df.columns else "Country or region"
for y, df in raw.items():
    print(y, "filas duplicadas:", df.duplicated().sum(), "| países repetidos en el año:", df[country_col(df)].duplicated().sum())

## 5. Consistencia de nombres de país entre años

In [ ]:
sets = {y: set(df[country_col(df)].str.strip()) for y, df in raw.items()}
common = set.intersection(*sets.values())
print("Países presentes en los 5 años:", len(common))
for y in YEARS:
    print(y, "no presentes en todos los años:", sorted(sets[y] - common)[:12], "...")

**Observación:** hay países que no aparecen en todos los años y algunos con nombres alternativos (p. ej. *Taiwan* vs *Taiwan Province of China*, *Hong Kong* vs *Hong Kong S.A.R., China*, *Trinidad & Tobago* vs *Trinidad and Tobago*, *North Cyprus* vs *Northern Cyprus*). Se estandarizan con un pequeño diccionario en el notebook de entrenamiento.

## 6. Tipos de dato y valores no numéricos

In [ ]:
num_like = ["Happiness Score","Happiness.Score","Score","Economy (GDP per Capita)","Economy..GDP.per.Capita.","GDP per capita",
            "Family","Social support","Health (Life Expectancy)","Health..Life.Expectancy.","Healthy life expectancy",
            "Freedom","Freedom to make life choices","Generosity","Trust (Government Corruption)","Trust..Government.Corruption.","Perceptions of corruption"]
for y, df in raw.items():
    cols = [c for c in df.columns if c in num_like]
    print(y, "todas numéricas:", all(pd.api.types.is_numeric_dtype(df[c]) for c in cols))

## 7. Outliers potenciales (regla IQR 1.5)

In [ ]:
rename = {
 "Country":"country","Country or region":"country",
 "Happiness Score":"happiness_score","Happiness.Score":"happiness_score","Score":"happiness_score",
 "Economy (GDP per Capita)":"gdp","Economy..GDP.per.Capita.":"gdp","GDP per capita":"gdp",
 "Family":"family","Social support":"family",
 "Health (Life Expectancy)":"health","Health..Life.Expectancy.":"health","Healthy life expectancy":"health",
 "Freedom":"freedom","Freedom to make life choices":"freedom",
 "Generosity":"generosity",
 "Trust (Government Corruption)":"corruption","Trust..Government.Corruption.":"corruption","Perceptions of corruption":"corruption"}
COLS = ["country","happiness_score","gdp","family","health","freedom","generosity","corruption"]
tmp = pd.concat([df.rename(columns=rename)[COLS].assign(year=y) for y, df in raw.items()], ignore_index=True)
num = ["happiness_score","gdp","family","health","freedom","generosity","corruption"]
rows = []
for c in num:
    q1, q3 = tmp[c].quantile([.25, .75]); iqr = q3 - q1
    n = ((tmp[c] < q1 - 1.5*iqr) | (tmp[c] > q3 + 1.5*iqr)).sum()
    rows.append((c, tmp[c].min(), tmp[c].max(), n))
pd.DataFrame(rows, columns=["variable","min","max","outliers_IQR"])

In [ ]:
fig, ax = plt.subplots(1, len(num), figsize=(18, 3.2))
for a, c in zip(ax, num):
    a.boxplot(tmp[c].dropna()); a.set_title(c)
plt.tight_layout(); plt.show()

**Observación:** los outliers se concentran en `corruption` y `generosity` (muchos países con valores bajos y unos pocos muy altos). Son **valores reales y plausibles**, no errores de captura → **no se eliminan**. Solo se descartarían valores negativos o imposibles.

## 8. Propuesta de esquema unificado

| Columna | Tipo | Origen (2015-2016 / 2017 / 2018-2019) |
|---|---|---|
| `country` | string | Country / Country / Country or region |
| `year` | int | nombre del archivo |
| `happiness_score` | float | Happiness Score / Happiness.Score / Score (**target**) |
| `gdp` | float | Economy (GDP per Capita) / Economy..GDP.per.Capita. / GDP per capita |
| `family` | float | Family / Family / Social support |
| `health` | float | Health (Life Expectancy) / Health..Life.Expectancy. / Healthy life expectancy |
| `freedom` | float | Freedom / Freedom / Freedom to make life choices |
| `generosity` | float | Generosity |
| `corruption` | float | Trust (Government Corruption) / Trust..Government.Corruption. / Perceptions of corruption |

**Columnas excluidas** (no existen en los 5 años o causan *target leakage*): `Region`, `Happiness Rank`, `Standard Error`, intervalos de confianza, `Whisker.high/low` y **`Dystopia Residual`** (el score es la suma de los 6 factores + dystopia residual, usarla filtraría el target). `Happiness Rank` también se excluye: es función directa del target.